# Logs, metrics and dashboards

Prometheus, Loki and Grafana, and the numbers MarginaliaRAG exports.

See the [README](README.md) for the why and the how.

In [ ]:
# Run from anywhere inside the repository: find its root so `marginalia` and the test helpers import,
# and read .env (the same settings the services use).
import os, sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'pyproject.toml').exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print(ROOT)

## Structured logs

In [ ]:
import io, logging
from marginalia import logs
out = io.StringIO()
logs.setup('notebook', fmt='json', stream=out)
token = logs.request_id.set('req-123')
logging.getLogger('marginalia.demo').info('question answered', extra={'question_id': 'q1', 'tokens': 9120})
logs.request_id.reset(token)
logs.setup('notebook', fmt='text')
print(out.getvalue())

## The metrics this process would export

In [ ]:
from marginalia import metrics
from prometheus_client import generate_latest
metrics.record_question('agent')
metrics.record_tokens({'agent': {'calls': 3, 'prompt': 9000, 'completion': 1200, 'total': 10200, 'seconds': 20.0, 'per_second': 60.0},
                       'total': {'calls': 3, 'prompt': 9000, 'completion': 1200, 'total': 10200, 'seconds': 20.0, 'per_second': 60.0}}, 'qwen3:4b')
print('\n'.join(l for l in generate_latest().decode().splitlines() if l.startswith('marginalia_') and '_bucket' not in l)[:2500])

## Prometheus and Loki (needs the observability profile)

In [ ]:
import httpx
q = lambda expr: httpx.get('http://127.0.0.1:9095/api/v1/query', params={'query': expr}).json()['data']['result']
print(q('up'))
print(q('sum by (outcome) (marginalia_questions_total)'))
print(q('sum by (role, kind) (marginalia_llm_tokens_total)'))

## The dashboards are generated
```bash
python docker/observability/build_dashboards.py
```

In [ ]:
import json, glob
for f in sorted(glob.glob('docker/observability/grafana/dashboards/*.json')):
    d = json.load(open(f, encoding='utf-8'))
    print(d['title'], '-', len(d['panels']), 'panels:', [p['title'] for p in d['panels'] if p['type'] != 'row'][:6])